# Tugas 2 — Modelling dan Evaluasi Model
## Klasifikasi Trafik Jaringan dengan Dataset UNSW-NB15

**Fokus:** klasifikasi biner `Normal (0)` dan `Attack (1)` menggunakan *supervised machine learning*.

Notebook ini merupakan **kelanjutan dari tugas EDA UNSW-NB15**. Tahapan yang dikerjakan adalah pemeriksaan ulang data → persiapan fitur → pembagian training–validation–testing → modelling → evaluasi → interpretasi → kesimpulan.

> **Audit terhadap notebook EDA sebelumnya:** output yang tersimpan menunjukkan `30.163` baris (45 kolom), dengan `30.162` label `0` dan satu label kosong, sedangkan teks penjelasannya menyebut `175.341` baris dan distribusi dua kelas. Data yang hanya berisi Normal **tidak dapat digunakan untuk melatih klasifikasi biner Normal/Attack**. Karena itu, notebook ini memakai pasangan file **UNSW_NB15_training-set.csv** dan **UNSW_NB15_testing-set.csv** yang lengkap; penggunaan data akan divalidasi sebelum pelatihan.

**Catatan:** kode dan metrik di bawah perlu dieksekusi di Google Colab. Angka performa tidak diisi sebelumnya atau direkayasa.

**Referensi dataset:** [UNSW Canberra: The UNSW-NB15 Dataset](https://research.unsw.edu.au/projects/unsw-nb15-dataset). Pembagian standar yang didokumentasikan oleh UNSW adalah **175.341 record training** dan **82.332 record testing**. Salinan untuk unduhan otomatis diambil dari repositori GitHub publik; bila salinan tidak tersedia, silakan gunakan opsi unggah file.

## A. Tujuan dan landasan teori

**Rumusan masalah:** bagaimana membangun model yang dapat mengklasifikasikan trafik jaringan sebagai Normal atau Attack, serta bagaimana menilai kemampuannya pada data yang belum pernah digunakan untuk melatih model?

**Algoritma yang dibandingkan:**
1. **Logistic Regression** — model klasifikasi linear yang mengestimasi probabilitas suatu data masuk ke kelas Attack.
2. **Random Forest** — algoritma *ensemble* yang menggabungkan prediksi dari banyak *decision tree* untuk menangkap hubungan yang lebih kompleks.
3. **Dummy Classifier** — *baseline* yang selalu memprediksi kelas mayoritas; dipakai sebagai pembanding agar akurasi yang tampak tinggi tidak langsung dianggap berkualitas.

**Desain evaluasi:** pasangan file UNSW-NB15 resmi sudah memisahkan **training** dan **testing**. Hanya data training yang dibagi lagi secara **stratified 80% training internal / 20% validation** untuk membandingkan model. Setelah pemilihan model berdasarkan **macro-F1 validation**, model terpilih dilatih kembali menggunakan **seluruh file training** dan dievaluasi **satu kali** pada file testing terpisah.

Mengapa demikian? *Preprocessing* (termasuk imputasi, encoding, dan scaling) harus dipelajari hanya dari data training agar tidak terjadi **data leakage**. File testing tidak digunakan memilih algoritma atau menyetel parameter.

## B. Persiapan environment dan dataset

Jalankan **Runtime → Run all** pada Google Colab.

**Opsi sumber data:** `DATA_SOURCE = "download"` untuk mengunduh dua CSV lengkap dari salinan publik, atau `DATA_SOURCE = "upload"` bila sudah mempunyai kedua CSV dan ingin mengunggahnya sendiri. Mode `download` **tidak memakai** file testing lokal 30.163 baris yang tidak lengkap.

In [ ]:
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

import sklearn

RANDOM_STATE = 42
DATA_SOURCE = "download"  # Ganti menjadi "upload" untuk mengunggah 2 file CSV sendiri
DATA_DIR = Path("/content") if Path("/content").exists() else Path.cwd()

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Lokasi kerja:", DATA_DIR)

In [ ]:
# Sumber unduhan: mirror publik dua berkas partisi UNSW-NB15.
# URL ini hanya untuk akses CSV; rujukan akademik utamanya tetap UNSW Canberra.
DATA_URLS = {
    "train": "https://raw.githubusercontent.com/Nir-J/ML-Projects/master/UNSW-Network_Packet_Classification/UNSW_NB15_training-set.csv",
    "test": "https://raw.githubusercontent.com/Nir-J/ML-Projects/master/UNSW-Network_Packet_Classification/UNSW_NB15_testing-set.csv",
}

TRAIN_FILE = DATA_DIR / "UNSW_NB15_training-set.csv"
TEST_FILE = DATA_DIR / "UNSW_NB15_testing-set.csv"

if DATA_SOURCE == "upload":
    try:
        from google.colab import files
        print("Unggah KEDUA file CSV: UNSW_NB15_training-set.csv dan UNSW_NB15_testing-set.csv")
        files.upload()
    except ImportError:
        print("Bukan Colab: letakkan kedua file CSV di folder:", DATA_DIR)
elif DATA_SOURCE == "download":
    import requests
    for kind, destination in (("train", TRAIN_FILE), ("test", TEST_FILE)):
        # File dengan nama yang sama selalu diperbarui di mode download,
        # sehingga tidak memakai file lokal lama yang mungkin terpotong.
        print(f"Mengunduh data {kind}...")
        try:
            with requests.get(DATA_URLS[kind], stream=True, timeout=120) as response:
                response.raise_for_status()
                with destination.open("wb") as fout:
                    for block in response.iter_content(chunk_size=1024 * 1024):
                        if block:
                            fout.write(block)
            print(f"  OK: {destination.name} ({destination.stat().st_size/1e6:.1f} MB)")
        except Exception as err:
            destination.unlink(missing_ok=True)
            raise RuntimeError(
                "Pengunduhan gagal. Ubah DATA_SOURCE='upload', lalu unggah dua berkas CSV "
                "UNSW-NB15 lengkap dari sumber dataset. Detail: " + str(err)
            ) from err
else:
    raise ValueError("DATA_SOURCE harus 'download' atau 'upload'.")

for path in (TRAIN_FILE, TEST_FILE):
    if not path.is_file():
        raise FileNotFoundError(f"File belum tersedia: {path}. Pastikan nama kedua file benar.")

### B.1 Membaca dan memvalidasi data

**Tujuan:** memastikan ada dua kelas pada file training dan testing, mendeteksi label yang hilang, serta menghentikan eksperimen jika dataset tidak cukup untuk klasifikasi biner. Pemeriksaan ini penting karena pada hasil EDA sebelumnya hanya kelas Normal yang terlihat.

In [ ]:
# utf-8-sig menangani karakter BOM (mis. pada nama kolom id).
def read_unsw(path):
    data = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    data.columns = data.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()
    if "label" not in data.columns:
        raise ValueError(f"File {path.name} tidak mempunyai kolom 'label'.")
    before = len(data)
    data["label"] = pd.to_numeric(data["label"], errors="coerce")
    # Hanya baris berlabel lengkap yang bisa digunakan dalam supervised learning.
    data = data.dropna(subset=["label"]).copy()
    removed = before - len(data)
    if removed:
        print(f"PERHATIAN: {path.name} memiliki {removed} baris label kosong/tidak valid yang dibuang.")
    labels = set(data["label"].unique())
    if labels != {0.0, 1.0}:
        raise ValueError(
            f"{path.name} hanya memiliki label {sorted(labels)}. "
            "Diperlukan kedua kelas 0=Normal dan 1=Attack. "
            "Data EDA lama yang hanya Normal tidak boleh dipakai untuk klasifikasi biner."
        )
    data["label"] = data["label"].astype(int)
    return data

train_df = read_unsw(TRAIN_FILE)
test_df = read_unsw(TEST_FILE)

print("Training:", train_df.shape, "| Testing:", test_df.shape)
print("\nDistribusi training:")
print(train_df["label"].value_counts().rename(index={0: "Normal", 1: "Attack"}))
print("\nDistribusi testing:")
print(test_df["label"].value_counts().rename(index={0: "Normal", 1: "Attack"}))

if len(train_df) != 175341 or len(test_df) != 82332:
    warnings.warn(
        "Ukuran file berbeda dari partisi UNSW-NB15 standar (175.341/82.332). "
        "Jika memakai versi subset, jelaskan jumlah aktual pada laporan.",
        stacklevel=1,
    )

display(train_df.head(3))

### B.2 Visualisasi distribusi kelas

Visualisasi membantu mengetahui ada tidaknya **class imbalance** (ketidakseimbangan kelas). Jika kelas Attack dan Normal mempunyai proporsi berbeda, *accuracy* saja bisa menyesatkan. Karena itu evaluasi juga memakai **precision, recall, F1-score, dan balanced accuracy**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, data) in zip(axes, (("Training", train_df), ("Testing", test_df))):
    counts = data["label"].value_counts().reindex([0, 1], fill_value=0)
    ax.bar(["Normal (0)", "Attack (1)"], counts.values, color=["#2C7FB8", "#F28E2B"])
    ax.set_title(f"Distribusi Label — {name}")
    ax.set_ylabel("Jumlah data")
    for i, v in enumerate(counts.values):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## C. Preprocessing — pemilihan fitur, imputasi, encoding, dan scaling

Langkah preprocessing:
- **Menghapus `label` dari fitur** karena itulah target yang diprediksi.
- **Menghapus `attack_cat`** karena nama kategori serangan dapat langsung membocorkan informasi target untuk tugas klasifikasi biner (*target leakage*).
- **Menghapus `id`** karena hanya pengenal baris, bukan pola trafik.
- **Fitur kategorikal** (`proto`, `service`, `state`, bila tersedia) → `SimpleImputer(most_frequent)` dan `OneHotEncoder(handle_unknown="ignore")`.
- **Fitur numerik** → `SimpleImputer(median)`. Untuk Logistic Regression ditambah `StandardScaler`, sedangkan Random Forest tidak memerlukan scaling numerik.

Seluruh transformasi dikemas dalam **`Pipeline` dan `ColumnTransformer`** agar parameter preprocessing dipelajari dari training saja, bukan dari validation/testing.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

TARGET = "label"
DROP_COLS = {"id", "attack_cat", "label"}

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]
missing_test_features = sorted(set(feature_cols) - set(test_df.columns))
if missing_test_features:
    raise ValueError(f"Fitur training tidak ditemukan di testing: {missing_test_features}")

X_full = train_df[feature_cols].copy()
y_full = train_df[TARGET].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET].copy()

categorical_cols = X_full.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

# Data numerik dengan nilai tak hingga ditangani oleh imputasi di dalam pipeline.
for col in numeric_cols:
    X_full[col] = pd.to_numeric(X_full[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")
X_full[numeric_cols] = X_full[numeric_cols].replace([np.inf, -np.inf], np.nan)
X_test[numeric_cols] = X_test[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Samakan dtype kategori dan isi yang tidak diketahui dibiarkan sebagai missing value.
for col in categorical_cols:
    X_full[col] = X_full[col].astype("string").astype("object")
    X_test[col] = X_test[col].astype("string").astype("object")
    X_full[col] = X_full[col].where(X_full[col].notna(), np.nan)
    X_test[col] = X_test[col].where(X_test[col].notna(), np.nan)

X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.20, stratify=y_full, random_state=RANDOM_STATE
)

print("Jumlah fitur input:", len(feature_cols))
print("Numerik:", len(numeric_cols), "| Kategorikal:", categorical_cols)
print("Train internal:", X_train.shape, "| Validation:", X_val.shape)
print("Test eksternal:", X_test.shape)
print("Fitur bocor dipakai?", bool(DROP_COLS.intersection(feature_cols)))
assert set(y_train.unique()) == {0, 1} and set(y_val.unique()) == {0, 1}

In [ ]:
# Dukungan untuk versi scikit-learn yang menggunakan 'sparse_output' atau 'sparse'.
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)

def build_preprocessor(scale_numeric=False):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    num_pipeline = Pipeline(num_steps)
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", one_hot_encoder),
    ])
    return ColumnTransformer([
        ("numeric", num_pipeline, numeric_cols),
        ("categorical", cat_pipeline, categorical_cols),
    ], remainder="drop")

## D. Modelling — membuat dan melatih model

**Logistic Regression** cocok sebagai pembanding sederhana dan mudah diinterpretasi. **Random Forest** dapat menangkap hubungan non-linear dan interaksi fitur. Keduanya menggunakan `class_weight` agar perhatian model tidak hanya tertuju pada kelas yang jumlahnya lebih banyak.

Penting: model dilatih hanya menggunakan **training internal**, sementara evaluasi awal dilakukan pada **validation**. `Random Forest` dapat membutuhkan beberapa menit bergantung lingkungan Colab.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay,
)

models = {
    "Logistic Regression": Pipeline([
        ("preprocess", build_preprocessor(scale_numeric=True)),
        ("classifier", LogisticRegression(
            max_iter=1000, solver="liblinear", class_weight="balanced",
            random_state=RANDOM_STATE
        )),
    ]),
    "Random Forest": Pipeline([
        ("preprocess", build_preprocessor(scale_numeric=False)),
        ("classifier", RandomForestClassifier(
            n_estimators=100, max_depth=20, min_samples_leaf=2,
            class_weight="balanced_subsample", random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
}

baseline = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
print("Model yang dievaluasi:", ", ".join(models.keys()))

### D.1 Fungsi evaluasi

Dalam *confusion matrix* biner dengan kelas positif = **Attack (1)**:
- **TP:** Attack terdeteksi sebagai Attack.
- **TN:** Normal terdeteksi sebagai Normal.
- **FP:** Normal keliru dinyatakan Attack (*false alarm*).
- **FN:** Attack keliru dinyatakan Normal (*serangan yang terlewat*).

Rumus evaluasi:

- **Accuracy** = `(TP + TN) / (TP + TN + FP + FN)`.
- **Precision Attack** = `TP / (TP + FP)`.
- **Recall Attack** = `TP / (TP + FN)`.
- **F1-score Attack** = `2 × Precision × Recall / (Precision + Recall)`.
- **Macro-F1** = rata-rata F1-score Normal dan Attack (memberi bobot sama untuk kedua kelas).
- **Balanced accuracy** = rata-rata recall kedua kelas.

**Interpretasi:** semakin tinggi recall Attack, semakin sedikit serangan yang terlewat, tetapi precision juga perlu diperhatikan karena prediksi serangan palsu dapat meningkatkan *false alarm*. Akurasi tinggi tanpa melihat confusion matrix belum cukup untuk menyatakan model berhasil.

In [ ]:
def evaluate_classifier(model, X, y, model_name):
    y_pred = model.predict(X)
    values = {
        "Model": model_name,
        "Accuracy": accuracy_score(y, y_pred),
        "Precision_Attack": precision_score(y, y_pred, pos_label=1, zero_division=0),
        "Recall_Attack": recall_score(y, y_pred, pos_label=1, zero_division=0),
        "F1_Attack": f1_score(y, y_pred, pos_label=1, zero_division=0),
        "F1_Macro": f1_score(y, y_pred, average="macro", zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y, y_pred),
    }
    if hasattr(model, "predict_proba"):
        classes = model.classes_
        positive_index = list(classes).index(1)
        y_scores = model.predict_proba(X)[:, positive_index]
        values["ROC_AUC"] = roc_auc_score(y, y_scores)
        values["PR_AUC_AP"] = average_precision_score(y, y_scores)
    return values

### D.2 Evaluasi pada data validation dan pemilihan model

Kita membandingkan Dummy Classifier, Logistic Regression, dan Random Forest pada **validation yang sama**. Pemilihan model akhir didasarkan pada **F1_Macro** tertinggi di validation, **bukan** pada hasil testing.

In [ ]:
validation_rows = []

baseline.fit(X_train, y_train)
validation_rows.append(evaluate_classifier(baseline, X_val, y_val, "Dummy (Mayoritas)"))

for name, model in models.items():
    print("Training:", name)
    model.fit(X_train, y_train)
    validation_rows.append(evaluate_classifier(model, X_val, y_val, name))
    print("Selesai:", name)

val_results = (pd.DataFrame(validation_rows)
                 .sort_values("F1_Macro", ascending=False)
                 .reset_index(drop=True))
display(val_results.style.format({col: "{:.4f}" for col in val_results.columns if col != "Model"}))

best_model_name = (val_results[val_results["Model"].isin(models.keys())]
                   .sort_values("F1_Macro", ascending=False)
                   .iloc[0]["Model"])
print("Model terpilih berdasarkan F1_Macro validation:", best_model_name)

In [ ]:
# Perbandingan hanya dilakukan pada VALIDATION, bukan memilih model dari TEST.
cols_to_plot = ["F1_Macro", "Recall_Attack", "Precision_Attack"]
ax = (val_results.set_index("Model")[cols_to_plot]
      .plot(kind="bar", figsize=(11, 5), ylim=(0, 1), rot=0))
ax.set_title("Perbandingan Model pada Validation")
ax.set_ylabel("Skor (0–1)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## E. Evaluasi akhir — test set terpisah

Model terpilih dilatih kembali menggunakan **seluruh file training**, kemudian dievaluasi pada **file testing yang tidak digunakan dalam pemilihan model**. Semua metrik berikut dihitung sungguh-sungguh pada saat kode dijalankan.

In [ ]:
# Buat ulang pipeline supaya model final benar-benar fit pada SELURUH data training.
if best_model_name == "Logistic Regression":
    final_model = Pipeline([
        ("preprocess", build_preprocessor(scale_numeric=True)),
        ("classifier", LogisticRegression(
            max_iter=1000, solver="liblinear", class_weight="balanced",
            random_state=RANDOM_STATE
        )),
    ])
else:
    final_model = Pipeline([
        ("preprocess", build_preprocessor(scale_numeric=False)),
        ("classifier", RandomForestClassifier(
            n_estimators=100, max_depth=20, min_samples_leaf=2,
            class_weight="balanced_subsample", random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

print("Melatih ulang model final:", best_model_name)
final_model.fit(X_full, y_full)
final_metrics = evaluate_classifier(final_model, X_test, y_test, best_model_name)
final_metrics_df = pd.DataFrame([final_metrics])
display(final_metrics_df.style.format({c: "{:.4f}" for c in final_metrics_df.columns if c != "Model"}))

final_predictions = final_model.predict(X_test)
final_probabilities = final_model.predict_proba(X_test)[:, list(final_model.classes_).index(1)]
print("\nClassification Report (Testing):")
print(classification_report(y_test, final_predictions,
                            labels=[0, 1], target_names=["Normal (0)", "Attack (1)"],
                            digits=4, zero_division=0))

### E.1 Confusion Matrix

Matriks dibaca dengan **baris = kelas sebenarnya** dan **kolom = kelas prediksi**. Beri perhatian khusus pada **FN** (Attack salah diprediksi Normal) serta **FP** (Normal salah diprediksi Attack). Di bidang keamanan jaringan, keduanya mempunyai konsekuensi yang berbeda.

In [ ]:
cm = confusion_matrix(y_test, final_predictions, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print(f"TN = {tn:,} | FP = {fp:,} | FN = {fn:,} | TP = {tp:,}")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(cm, display_labels=["Normal", "Attack"]).plot(
    ax=ax, cmap="Blues", values_format="d", colorbar=False
)
ax.set_title(f"Confusion Matrix — {best_model_name} (Testing)")
plt.tight_layout()
plt.show()

### E.2 ROC Curve dan Precision–Recall Curve (pelengkap)

- **ROC Curve** menunjukkan hubungan *true positive rate* (recall) dan *false positive rate* pada berbagai ambang keputusan. Semakin mendekati kiri-atas umumnya semakin baik; **ROC-AUC** merangkum performa lintas threshold.
- **Precision–Recall Curve** menunjukkan kompromi precision dan recall untuk kelas Attack. Ini membantu ketika biaya salah deteksi tidak seimbang.

Kedua grafik ini **pelengkap**, bukan pengganti metrik berbasis threshold dan confusion matrix.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y_test, final_probabilities, ax=axes[0], name=best_model_name)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.7)
axes[0].set_title("ROC Curve — Testing")
PrecisionRecallDisplay.from_predictions(y_test, final_probabilities,
                                        ax=axes[1], name=best_model_name)
axes[1].set_title("Precision–Recall Curve — Testing")
plt.tight_layout()
plt.show()

## F. Penjelasan hasil dan pembahasan (otomatis, berdasarkan output asli)

Sel ini menghasilkan uraian yang dapat dijadikan dasar laporan tugas. **Salin setelah semua sel dijalankan**, bukan sebelum hasil tersedia. Penafsiran tidak boleh mengklaim model mampu mendeteksi serangan dunia nyata secara sempurna; hasil hanya berlaku pada pengaturan pengujian UNSW-NB15 yang digunakan.

In [ ]:
acc = final_metrics["Accuracy"]
prec = final_metrics["Precision_Attack"]
rec = final_metrics["Recall_Attack"]
f1 = final_metrics["F1_Attack"]
f1m = final_metrics["F1_Macro"]
bacc = final_metrics["Balanced_Accuracy"]
auc = final_metrics["ROC_AUC"]

if fn > fp:
    error_note = "Jumlah FN lebih banyak daripada FP: perhatian utama adalah serangan yang tidak terdeteksi."
elif fp > fn:
    error_note = "Jumlah FP lebih banyak daripada FN: perhatian utama adalah alarm palsu pada trafik normal."
else:
    error_note = "Jumlah FP dan FN sama; keduanya tetap perlu dianalisis terhadap kebutuhan sistem."

report = f"""
### Pembahasan hasil Modelling dan Evaluasi

**1. Data dan target.** Penelitian menggunakan UNSW-NB15 untuk klasifikasi biner: 0 = Normal dan 1 = Attack. Data training berjumlah **{len(train_df):,}** baris dan data testing berjumlah **{len(test_df):,}** baris. Sebanyak **{len(feature_cols)} fitur** digunakan, sedangkan `id` dan `attack_cat` tidak dipakai sebagai prediktor untuk menghindari informasi pengenal dan kebocoran target.

**2. Preprocessing.** Fitur numerik ditangani menggunakan imputasi median, sementara fitur kategorikal menggunakan imputasi modus dan one-hot encoding. Standardisasi ditambahkan pada Logistic Regression. Seluruh proses dilakukan dalam pipeline yang di-fit pada training sehingga tidak mengambil informasi dari validation atau testing.

**3. Modelling.** Logistic Regression dan Random Forest dibandingkan dengan Dummy Classifier sebagai baseline. Model terbaik berdasarkan **macro-F1 validation** adalah **{best_model_name}**. Setelah pemilihan, model tersebut dilatih ulang pada seluruh training set.

**4. Evaluasi testing.** Model menghasilkan **accuracy {acc:.2%}**, **precision Attack {prec:.2%}**, **recall Attack {rec:.2%}**, **F1-score Attack {f1:.2%}**, **macro-F1 {f1m:.2%}**, **balanced accuracy {bacc:.2%}**, serta **ROC-AUC {auc:.4f}**. Metrik-metrik ini saling melengkapi sehingga kualitas model tidak disimpulkan hanya dari accuracy.

**5. Confusion Matrix.** Terdapat **{tn:,} TN**, **{fp:,} FP**, **{fn:,} FN**, dan **{tp:,} TP**. FP adalah trafik Normal yang salah dideteksi sebagai Attack; FN adalah trafik Attack yang terlewat. **{error_note}**

**6. Kesimpulan.** Pada pengujian dengan file testing UNSW-NB15, model **{best_model_name}** menghasilkan macro-F1 **{f1m:.2%}**. Performa tersebut hanya mencerminkan data dan rancangan uji ini, belum membuktikan generalisasi ke seluruh trafik jaringan dunia nyata. Pengembangan berikutnya dapat berupa validasi lintas dataset, analisis jenis serangan, dan penyetelan threshold menggunakan data validation (bukan testing).
"""
display(Markdown(report))

## G. Menyimpan hasil tugas

Tabel evaluasi dan rangkuman penjelasan disimpan sebagai file. Untuk mengumpulkan tugas di Colab, pastikan notebook sudah dijalankan sehingga **output tabel, grafik, dan penjelasan** terlihat, lalu simpan melalui menu **File → Download → Download .ipynb**. Jangan mengumpulkan notebook yang hanya berisi kode tanpa output jika dosen meminta hasil eksperimen.

In [ ]:
validation_path = DATA_DIR / "Tugas2_Hasil_Validation.csv"
testing_path = DATA_DIR / "Tugas2_Hasil_Testing.csv"
report_path = DATA_DIR / "Tugas2_Pembahasan_Hasil.md"
val_results.to_csv(validation_path, index=False)
final_metrics_df.to_csv(testing_path, index=False)
report_path.write_text(report, encoding="utf-8")
print("Hasil disimpan:")
print(" -", validation_path)
print(" -", testing_path)
print(" -", report_path)

## H. Kesimpulan pembelajaran

- **Modelling** adalah tahap membangun pola prediksi dari data berlabel menggunakan algoritma klasifikasi.
- **Validation** dipakai untuk membandingkan model dan memilih metode, sedangkan **testing** mengukur performa akhir pada data yang dipisahkan sejak awal.
- **Accuracy** menunjukkan proporsi prediksi yang benar, tetapi **precision**, **recall**, **F1-score**, dan **confusion matrix** diperlukan untuk melihat jenis kesalahan model.
- **Data leakage** harus dihindari dengan menghapus `attack_cat`/`label` dari prediktor dan hanya mempelajari preprocessing dari training.
- **Keterbatasan:** dataset merupakan data uji terkontrol; performa pada jaringan nyata mungkin berbeda. Karena hanya dua algoritma dasar dan parameter terbatas yang diuji, hasil belum mewakili seluruh algoritma maupun optimasi yang mungkin.

**Daftar pustaka:** Moustafa, N., & Slay, J. (2015). *UNSW-NB15: A comprehensive data set for network intrusion detection systems (UNSW-NB15 network data set).* MilCIS. Sumber dataset: https://research.unsw.edu.au/projects/unsw-nb15-dataset

**Catatan kesesuaian materi:** jika dosen memberi slide materi minggu ini yang menetapkan algoritma/rasio split/metrik tertentu, sesuaikan bagian-bagian itu dengan instruksi resmi. Notebook ini memakai pendekatan standar klasifikasi terawasi dan evaluasi model.